# 00 · Сбор и подготовка данных

Семь стадий: котировки, корпоративные события и состав индекса с MOEX ISS → новости из Telegram →
привязка новостей к тикерам → ruBERT-эмбеддинги → таргет и ценовые признаки → матрица признаков
узлов → динамический граф. Каждая стадия читает и пишет файлы в `data/`, поэтому стадии можно
перезапускать по отдельности.

Папка `data/` в репозиторий не входит. Стадии 1–2 требуют сетевого доступа (MOEX ISS, Telegram API
с ключами из `.env`), стадия 4 требует GPU/MPS (≈30 минут). Стадии 3, 5, 6, 7 детерминированы
и пересчитываются из результатов стадий 1, 2 и 4 за несколько минут на CPU.

In [1]:
import re, os, time, bisect, itertools
from pathlib import Path
import numpy as np, pandas as pd

DATA = Path('data'); DATA.mkdir(exist_ok=True)

## 1. Состав IMOEX, дневные котировки и корпоративные события (MOEX ISS)

Открытый API без ключей. Сектор эмитента берётся из состава отраслевых индексов Мосбиржи; пустой
состав индекса считается ошибкой, а не молча превращается в сектор `Other`. Состав индекса
фиксируется при первой выгрузке и при повторных запусках берётся из `tickers.csv`; секторы
пересчитываются по текущему составу отраслевых индексов. В работе используется выгрузка весны
2026 года: 46 бумаг индекса после мартовской ребалансировки. Состав выбран после окончания периода
данных, это survivorship bias; его масштаб оценивается по фактическому составу индекса ниже.

Котировки ISS не скорректированы, поэтому отдельно выгружаются сплиты/консолидации и, если ISS их
отдаёт, дивиденды: по ним на стадии 5 считается скорректированная доходность. На момент последней
сборки (октябрь 2026) публичный ISS блок дивидендов без авторизации не отдаёт, поэтому доходности
в работе ценовые: скорректированы на сплиты, но не на дивиденды.

In [2]:
import requests
BASE = 'https://iss.moex.com/iss'
SECTOR_INDICES = {'MOEXOG': 'Oil & Gas', 'MOEXEU': 'Electric Utilities', 'MOEXMM': 'Metals & Mining',
                  'MOEXTL': 'Telecom', 'MOEXFN': 'Financials', 'MOEXCN': 'Consumer', 'MOEXCH': 'Chemicals',
                  'MOEXRE': 'Real Estate', 'MOEXIT': 'IT', 'MOEXTN': 'Transport'}
HISTORY_FROM, HISTORY_TILL = '2022-08-01', '2025-12-31'
HISTORY_COLS = 'TRADEDATE,OPEN,HIGH,LOW,CLOSE,VOLUME,VALUE'


session = requests.Session()                            # одно keep-alive соединение на все запросы


def get(url, params, retries=6):
    for attempt in range(retries):                      # MOEX иногда рвёт соединение, повторяем с паузой
        try:
            r = session.get(url, params=params, timeout=30)
        except requests.RequestException:
            if attempt == retries - 1: raise
            time.sleep(2 ** attempt); continue
        if r.status_code >= 500 and attempt < retries - 1:   # 4xx не повторяем: это ошибка самого запроса
            time.sleep(2 ** attempt); continue
        r.raise_for_status(); return r.json()


def table(js, block):
    return pd.DataFrame(js[block]['data'], columns=[c.lower() for c in js[block]['columns']])


def index_members(index_id):
    js = get(f'{BASE}/statistics/engines/stock/markets/index/analytics/{index_id}.json',
             {'iss.meta': 'off', 'analytics.columns': 'ticker', 'limit': 100})
    members = [r[0] for r in js.get('analytics', {}).get('data', []) if r[0]]
    assert members, f'индекс {index_id} вернул пустой состав: проверьте код индекса'
    return members


def fetch_history(secid):
    url = f'{BASE}/history/engines/stock/markets/shares/boards/TQBR/securities/{secid}.json'
    rows, start = [], 0
    while True:                                         # пагинация через history.cursor
        js = get(url, {'iss.meta': 'off', 'history.columns': HISTORY_COLS,
                       'from': HISTORY_FROM, 'till': HISTORY_TILL, 'start': start})
        chunk = js['history']['data']
        if not chunk: break
        rows += chunk
        index, total, pagesize = js['history.cursor']['data'][0]
        if index + pagesize >= total: break
        start = index + pagesize; time.sleep(0.3)
    return pd.DataFrame(rows, columns=HISTORY_COLS.split(',')).assign(SECID=secid)


if (DATA / 'tickers.csv').exists():                    # состав фиксируется при первой выгрузке; для текущего состава удалите файл
    tickers = pd.read_csv(DATA / 'tickers.csv').drop(columns='SECTOR')
else:
    js = get(f'{BASE}/engines/stock/markets/shares/boards/TQBR/securities.json',
             {'iss.meta': 'off', 'securities.columns': 'SECID,SHORTNAME,ISIN,LOTSIZE,FACEVALUE,PREVPRICE'})
    tickers = pd.DataFrame(js['securities']['data'], columns=js['securities']['columns'])
    tickers = tickers[tickers['SECID'].isin(index_members('IMOEX'))].reset_index(drop=True)
sector = {}
for index_id, name in SECTOR_INDICES.items():           # бумага из нескольких индексов получает первый по порядку словаря
    for t in index_members(index_id):
        sector.setdefault(t, name)
SECTOR_OVERRIDE = {'PIKK': 'Real Estate'}                # девелопер, которого нет в текущем составе MOEXRE
tickers['SECTOR'] = tickers['SECID'].map({**sector, **SECTOR_OVERRIDE}).fillna('Other')
tickers.to_csv(DATA / 'tickers.csv', index=False)
print(tickers['SECTOR'].value_counts().to_string())

history = pd.concat([fetch_history(s) for s in tickers['SECID']], ignore_index=True)
history['TRADEDATE'] = pd.to_datetime(history['TRADEDATE'])
history.sort_values(['SECID', 'TRADEDATE']).to_csv(DATA / 'history_ohlcv.csv', index=False)

splits = table(get(f'{BASE}/statistics/engines/stock/splits.json', {'iss.meta': 'off'}), 'splits')
splits = splits[splits['secid'].isin(tickers['SECID'])]
splits.to_csv(DATA / 'splits.csv', index=False)
def dividends(secid):
    js = get(f'{BASE}/securities/{secid}/dividends.json', {'iss.meta': 'off'})
    return table(js, 'dividends') if 'dividends' in js else None


if dividends(tickers['SECID'][0]) is not None:
    divs = pd.concat([dividends(s) for s in tickers['SECID']], ignore_index=True)
    divs.to_csv(DATA / 'dividends.csv', index=False)
    print(f'{len(divs)} дивидендных записей')
else:                                                   # публичный ISS перестал отдавать блок dividends без авторизации
    print('ВНИМАНИЕ: ISS не отдаёт дивиденды, dividends.csv не создан; доходности будут ценовыми (без дивидендов)')
print(f'{len(tickers)} тикеров, {len(history):,} строк котировок, {len(splits)} сплитов')

SECTOR
Financials            10
Oil & Gas              9
Metals & Mining        9
IT                     6
Consumer               3
Electric Utilities     2
Transport              2
Telecom                2
Other                  1
Chemicals              1
Real Estate            1


ВНИМАНИЕ: ISS не отдаёт дивиденды, dividends.csv не создан; доходности будут ценовыми (без дивидендов)
46 тикеров, 35,754 строк котировок, 5 сплитов


### История состава IMOEX

Выборка — 46 бумаг из состава на декабрь 2025 года. Чтобы оценить, сколько это стоит
(survivorship bias), выгружается фактический состав индекса на первый торговый день каждой
недели периода: ISS отдаёт его по дате. Состав пересматривается раз в квартал, поэтому недельных
снимков достаточно. На стадиях 5–7 состав не используется; он нужен для проверки в ноутбуках 01 и 03.

In [3]:
def index_snapshot(index_id, date):
    js = get(f'{BASE}/statistics/engines/stock/markets/index/analytics/{index_id}.json',
             {'iss.meta': 'off', 'date': date, 'limit': 100, 'analytics.columns': 'tradedate,secids,weight'})
    return table(js, 'analytics')


cal = pd.DatetimeIndex(np.sort(pd.read_csv(DATA / 'history_ohlcv.csv', usecols=['TRADEDATE'],
                                           parse_dates=['TRADEDATE'])['TRADEDATE'].unique()))
weeks = cal.to_series().groupby(cal.to_period('W')).min()           # первый торговый день недели
comp = pd.concat([index_snapshot('IMOEX', d.strftime('%Y-%m-%d')) for d in weeks], ignore_index=True)
assert comp['tradedate'].nunique() == len(weeks), 'на часть дат ISS не вернул состав'
comp = comp.rename(columns={'tradedate': 'DATE', 'secids': 'SECID', 'weight': 'WEIGHT'})
comp.to_csv(DATA / 'imoex_composition.csv', index=False)
out = ~comp['SECID'].isin(tickers['SECID'])
print(f'{len(weeks)} снимков состава; бумаг в индексе за период {comp.SECID.nunique()}, '
      f'из них вне выборки {comp.SECID[out].nunique()}')

179 снимков состава; бумаг в индексе за период 66, из них вне выборки 21


## 2. Новостной корпус Telegram

Три канала разных жанров: брокер (`bcs_express`), медиа (`RBCinvest`), агрегатор
(`smartlabnews`). Учётные данные читаются из `.env` (`TELEGRAM_API_ID`, `TELEGRAM_API_HASH`,
`TELEGRAM_PHONE`, необязательно `TELEGRAM_SESSION_NAME`); файл сессии Telethon в репозиторий
не попадает. Сообщения идут от новых к старым, цикл обрывается на первой дате раньше начала периода.

In [ ]:
from telethon import TelegramClient

if Path('.env').exists():                                # TELEGRAM_API_ID / TELEGRAM_API_HASH / TELEGRAM_PHONE
    for line in Path('.env').read_text().splitlines():
        k, _, v = line.strip().partition('=')
        if k and not k.startswith('#'):
            os.environ.setdefault(k.strip(), v.strip().strip('\'"'))
CHANNELS = ['bcs_express', 'RBCinvest', 'smartlabnews']
START, END = pd.Timestamp('2022-08-01', tz='UTC'), pd.Timestamp('2026-01-01', tz='UTC')


async def fetch(client, channel):
    rows = []
    async for m in client.iter_messages(channel, offset_date=END):
        if m.date < START: break
        if m.text:
            rows.append(dict(channel=channel, date=m.date, text=m.text, views=m.views, forwards=m.forwards))
    return rows


async with TelegramClient(os.getenv('TELEGRAM_SESSION_NAME', 'news_session'),
                          int(os.environ['TELEGRAM_API_ID']), os.environ['TELEGRAM_API_HASH']) as client:
    await client.start(phone=os.environ['TELEGRAM_PHONE'])
    news = pd.DataFrame([r for c in CHANNELS for r in await fetch(client, c)])
news.to_csv(DATA / 'news_aug2022_dec2025.csv', index=False)
print(news['channel'].value_counts())

## 3. Привязка новостей к тикерам

Дедупликация репостов по нормализованным первым 200 символам, затем трёхуровневый каскад:
явное упоминание эмитента (вес 1,0) → отраслевое ключевое слово, всем тикерам сектора (0,5) →
иначе всему индексу (0,2). Уровни взаимоисключающие.

Перед поиском текст приводится к нижнему регистру, «ё» заменяется на «е», а упоминания индексов
(«индекс Мосбиржи», IMOEX, RTS), биржи как площадки («торги на Мосбирже», «срочный рынок Мосбиржи»),
ссылки и названия соцсетей («YouTube и ВК») вырезаются: иначе обзор рынка приписывался бы тикеру
MOEX, а анонс видео — VKCO. Каждый
шаблон начинается с границы слова `\b`, поэтому «найти» не ловит «айти», «экспортов» не ловит
«портов», а «труда» не ловит «руда»; окончания допускаются только там, где имя склоняется.
Синонимы заданы для всех тикеров текущего состава, привилегированные акции получают те же
новости, что и обыкновенные.

In [4]:
news = pd.read_csv(DATA / 'news_aug2022_dec2025.csv', parse_dates=['date'])
tickers = pd.read_csv(DATA / 'tickers.csv')

key = news['text'].map(lambda t: ' '.join(t.lower().split())[:200] if isinstance(t, str) else '')
news = (news.assign(_k=key).sort_values('date').drop_duplicates('_k', keep='first')
            .drop(columns='_k').reset_index(drop=True))
news['news_id'] = news.index
print(f'после дедупликации: {len(news):,} новостей')

SYNONYMS = {   # шаблоны в нижнем регистре и с «е» вместо «ё»
    'AFKS': [r'\bафк\W{0,3}систем\w*', r'\bsistema\b'],
    'AFLT': [r'\bаэрофлот\w*', r'\baeroflot\b'],
    'ALRS': [r'\bалрос\w*', r'\balrosa\b'],
    'BSPB': [r'\bбанк\w*\s+санкт-петербург\w*', r'\bбсп\b', r'\bbspb\b'],
    'CBOM': [r'\bмкб\b', r'\bмосковск\w+\s+кредитн\w+\s+банк\w*'],
    'CHMF': [r'\bсеверстал\w*', r'\bseverstal\b'],
    'CNRU': [r'\bциан(?:а|у|ом|е)?\b', r'\bcian\b'],
    'DOMRF': [r'\bдом\W?рф\b'],
    'ENPG': [r'\bэн\+', r'\ben\+', r'\bэн\s?плюс\b'],
    'FLOT': [r'\bсовкомфлот\w*', r'\bsovcomflot\b'],
    'GAZP': [r'\bгазпром(?!\s*нефт|банк|\w*\s+нефт)\w*', r'\bgazprom\b(?!\s*neft)'],
    'GMKN': [r'\bнорникел\w*', r'\bнорильск\w*\s+никел\w*', r'\bnornickel\b', r'\bгмк\b'],
    'HEAD': [r'\bхедхантер\w*', r'\bхэдхантер\w*', r'\bheadhunter\b', r'\bhh\.ru\b'],
    'IRAO': [r'\bинтер\s?рао\b', r'\binter\s?rao\b'],
    'LENT': [r'«лент[аыеу]»', r'\b(?:ритейлер|сет|гипермаркет)\w*\s+«?лент[аыеу]\b', r'\blenta\b'],
    'LKOH': [r'\bлукойл\w*', r'\blukoil\b'],
    'MAGN': [r'\bммк\b', r'\bмагнитогорск\w*\s+металлург\w*', r'\bmmk\b'],
    'MDMG': [r'\bмать\s+и\s+дитя\b', r'\bmd\s?medical\b', r'\bmdmg\b'],
    'MOEX': [r'\bмосбирж\w*', r'\bмосковск\w+\s+бирж\w*', r'\bmoex\b'],
    'MSNG': [r'\bмосэнерго\w*'],
    'MTSS': [r'\bмтс\b(?!\W{0,2}банк)', r'\bmts\b'],
    'NLMK': [r'\bнлмк\b', r'\bnlmk\b'],
    'NVTK': [r'\bноватэк\w*', r'\bnovatek\b'],
    'OZON': [r'\bозон(?:а|у|ом|е)?\b', r'\bozon\b'],
    'PHOR': [r'\bфосагро\w*', r'\bphosagro\b'],
    'PIKK': [r'\b(?:гк|групп\w*)\s+«?пик\b', r'«пик»', r'\bpik\s+group\b'],
    'PLZL': [r'\bполюс(?:а|у|ом|е)?\b', r'\bpolyus\b'],
    'POSI': [r'\bpositive\s+technologies\b', r'\bпозитив\s+текнолоджиз\b', r'«позитив\w*»', r'\bгрупп\w*\s+«?позитив\b'],
    'RENI': [r'\bренессанс\s+страхован\w*', r'\bгрупп\w*\s+«?ренессанс\b'],
    'ROSN': [r'\bроснефт\w*', r'\brosneft\b'],
    'RTKM': [r'\bростелеком\w*', r'\brostelecom\b'],
    'RUAL': [r'\bрусал\w*', r'\brusal\b'],
    'SBER': [r'\bсбербанк\w*', r'\bсбер(?:а|у|ом|е)?\b', r'\bsber\w*'],
    'SNGS': [r'\bсургутнефтегаз\w*', r'\bсургутнг\b', r'\bsurgutneftegas\b'],
    'SVCB': [r'\bсовкомбанк\w*', r'\bsovcombank\b'],
    'T': [r'\bт-банк\w*', r'\bт-технолог\w*', r'\bтинькофф\w*', r'\btinkoff\b', r'\btbank\b'],
    'TATN': [r'\bтатнефт\w*', r'\btatneft\b'],
    'TRNFP': [r'\bтранснефт\w*', r'\btransneft\b'],
    'UGLD': [r'\bюгк\b', r'\bюжуралзолото\w*'],
    'VKCO': [r'\bвконтакте\b', r'\bвк\b', r'\bvk\b'],
    'VTBR': [r'\bвтб\b', r'\bvtb\b'],
    'X5': [r'\bx5\b', r'\bикс\s?5\b', r'\bпятерочк\w*', r'\bперекрест(?:ок|ка|ке|ку)\b'],
    'YDEX': [r'\bяндекс\w*', r'\byandex\b'],
}
PREFERRED = {'SBERP': 'SBER', 'SNGSP': 'SNGS', 'TATNP': 'TATN'}   # «префы» получают новости обыкновенных акций
SECTOR_KEYWORDS = {
    'Oil & Gas': [r'\bнефт\w*', r'\bгаз\b', r'\bгазов\w*', r'\bопек\b', r'\bopec\b'],
    'Metals & Mining': [r'\bметаллург\w*', r'\bсталелит\w*', r'\bруд[аыуе]\b', r'\bзолотодоб\w*', r'\bникел\w*',
                        r'\bалюмини\w*', r'\bдобыч\w*.{0,5}мет'],
    'Financials': [r'\bбанковск\w+\s+сектор\w*', r'\bбанк(?:и|ам|ов)\b', r'\bкредитн\w*', r'\bбанкир\w*'],
    'Electric Utilities': [r'\bэлектроэнерг\w*', r'\bэнергетик\w*', r'\bгенерирующ\w*'],
    'Telecom': [r'\bтелеком\w*', r'\bоператор\w*\s+связи\b', r'\bсотов\w+\s+связи\b'],
    'IT': [r'\bит-\w+', r'\bайти\b', r'\bхайтек\w*', r'\bсофтвер\w*', r'\bтехнологическ\w+\s+сектор\w*',
           r'\bкибербезопасност\w*'],
    'Consumer': [r'\bритейл\w*', r'\bрозничн\w*\s+торгов\w*', r'\bfmcg\b'],
    'Real Estate': [r'\bдевелопер\w*', r'\bжилищ\w*', r'\bзастройщик\w*', r'\bнедвижим\w*'],
    'Chemicals': [r'\bхимическ\w*', r'\bудобрени\w*'],
    'Transport': [r'\bавиаперевоз\w*', r'\bавиакомпани\w*', r'\bгрузоперевоз\w*', r'\bпорт(?:ы|ов|ам|ах)?\b',
                  r'\bтанкер\w*'],
}
EXCHANGE = r'(?:мосбирж\w*|московск\w+\s+бирж\w*)'
MARKET_RE = re.compile(                                   # упоминания индексов, биржи как площадки, ссылок и соцсетей
    r'https?://\S+|\b[\w.-]+\.(?:com|ru|me)/\S*'
    rf'|\bиндекс\w*\s+«?(?:{EXCHANGE}|imoex\w*|moex\w*|ммвб|ртс|rts)»?|\bimoex\w*|\bmoexbc\b|\brts\b'
    rf'|\bна\s+(?:(?:срочном|валютном|фондовом|денежном)\s+)?(?:рынк\w*\s+)?{EXCHANGE}|\bрынк\w*\s+{EXCHANGE}'
    r'|\b(?:youtube|ютуб\w*|rutube|telegram|телеграм\w*)\s+и\s+(?:вк|vk)\b|\b(?:вк|vk)\s+видео\b')
imoex = set(tickers['SECID'])
SYNONYMS.update({p: SYNONYMS[o] for p, o in PREFERRED.items() if o in SYNONYMS})
print('тикеры индекса без синонимов:', sorted(imoex - set(SYNONYMS)) or 'нет')
print('синонимы для тикеров вне индекса:', sorted(set(SYNONYMS) - imoex) or 'нет')
print('сектора без тикеров:', sorted(set(SECTOR_KEYWORDS) - set(tickers['SECTOR'])) or 'нет')
TICKER_RE = {t: re.compile('|'.join(p)) for t, p in SYNONYMS.items() if t in imoex}
SECTOR_RE = {s: re.compile('|'.join(p)) for s, p in SECTOR_KEYWORDS.items()}
sector_tickers = tickers.groupby('SECTOR')['SECID'].apply(list).to_dict()


def link(news_id, text):
    if not isinstance(text, str):
        return []
    text = MARKET_RE.sub(' ', text.lower().replace('ё', 'е'))
    hit = [t for t, r in TICKER_RE.items() if r.search(text)]
    if hit:
        return [(news_id, t, 1.0, 1) for t in hit]
    hit = dict.fromkeys(t for s, r in SECTOR_RE.items() if r.search(text) for t in sector_tickers.get(s, []))
    if hit:
        return [(news_id, t, 0.5, 2) for t in hit]
    return [(news_id, t, 0.2, 3) for t in tickers['SECID']]


linked = pd.DataFrame([row for i, t in zip(news['news_id'], news['text']) for row in link(i, t)],
                      columns=['news_id', 'SECID', 'weight', 'level'])
linked = (linked.merge(news[['news_id', 'date', 'channel', 'text', 'views', 'forwards']], on='news_id')
                .sort_values(['SECID', 'date']).reset_index(drop=True))
linked.to_csv(DATA / 'news_linked.csv', index=False)
print(linked.groupby('level').agg(связей=('news_id', 'size'), новостей=('news_id', 'nunique')))
print(linked.query('level == 1')['SECID'].value_counts().to_string())

после дедупликации: 22,345 новостей
тикеры индекса без синонимов: нет
синонимы для тикеров вне индекса: нет
сектора без тикеров: нет


       связей  новостей
level                  
1       25535      9522
2       25926      2983
3      452640      9840
SECID
SBERP    1710
SBER     1710
MOEX     1702
GAZP     1578
LKOH     1097
VTBR     1030
YDEX      819
VKCO      755
NVTK      746
TATN      739
TATNP     739
ROSN      713
SNGSP     670
SNGS      670
GMKN      660
PLZL      647
OZON      625
CHMF      577
MTSS      541
X5        476
ALRS      476
POSI      460
TRNFP     456
PHOR      441
AFLT      440
RUAL      400
T         388
MAGN      383
RTKM      381
UGLD      377
NLMK      366
HEAD      345
AFKS      344
FLOT      309
BSPB      273
IRAO      263
SVCB      256
MDMG      187
CNRU      182
CBOM      169
ENPG      133
MSNG      103
DOMRF      82
RENI       78
PIKK       22
LENT       17


## 4. ruBERT-эмбеддинги

`DeepPavlov/rubert-base-cased`, mean-pooling по последнему скрытому слою с учётом маски,
768 компонент. Считается один раз на уникальную новость и кэшируется; агрегация по узлам
делается на следующей стадии.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL = 'DeepPavlov/rubert-base-cased'
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
tok, model = AutoTokenizer.from_pretrained(MODEL), AutoModel.from_pretrained(MODEL).to(device).eval()

uniq = (pd.read_csv(DATA / 'news_linked.csv', usecols=['news_id', 'text', 'date', 'channel'])
          .drop_duplicates('news_id').sort_values('news_id').reset_index(drop=True))
uniq['text'] = uniq['text'].fillna('').astype(str)


@torch.no_grad()
def embed(texts):
    enc = tok(texts, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
    mask = enc['attention_mask'].unsqueeze(-1).float()
    h = model(**enc).last_hidden_state
    return ((h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)).cpu().numpy().astype(np.float32)


emb = np.vstack([embed(uniq['text'][i:i + 32].tolist()) for i in range(0, len(uniq), 32)])
np.save(DATA / 'news_embeddings.npy', emb)
uniq[['news_id', 'date', 'channel']].to_csv(DATA / 'news_embeddings_index.csv', index=False)
print(emb.shape)

## 5. Скорректированные цены, таргет и ценовые признаки

Котировки ISS сырые, поэтому сначала строится скорректированный ряд. Сплиты и консолидации берутся
из `splits.csv`; дополнительно любой дневной скачок цены больше чем в 3 раза считается
корпоративным событием, и его коэффициент округляется до «круглого» (1:100, 5000:1 и т. п.).
Цены до события делятся на коэффициент, объёмы умножаются. Дивиденды из `dividends.csv` (если
файл есть) начисляются в дату отсечки: первый торговый день, сделки которого рассчитываются после даты
закрытия реестра (T+2, с 31.07.2023 T+1).

`RETURN(t)` — доходность от закрытия t−1 до закрытия t с учётом дивиденда, если он есть в `dividends.csv`; если бумага пропустила торговую
сессию рынка, доходность через пропуск не считается. `FUTURE_RETURN(t) = RETURN(t+1)`;
трёхклассовая разметка по терцилям делается позже, внутри каждого фолда, только по его train-части.
Ценовые признаки: доходности дней t, t−1, …, t−4 (`RET_LAG0` — доходность самого дня t, она
известна на закрытии) и логарифм отношения объёма к его 20-дневному среднему.

In [5]:
df = (pd.read_csv(DATA / 'history_ohlcv.csv', parse_dates=['TRADEDATE'])
        .query('CLOSE > 0').sort_values(['SECID', 'TRADEDATE']).reset_index(drop=True))   # строки без сделок выбрасываем
cal = np.sort(df['TRADEDATE'].unique())                                                    # торговый календарь рынка
df['DAY'] = np.searchsorted(cal, df['TRADEDATE'].values)
g = df.groupby('SECID')
gap = df['DAY'] - g['DAY'].shift(1)
log_jump = np.log(df['CLOSE'] / g['CLOSE'].shift(1))

# сплиты и консолидации: ratio = новых бумаг за одну старую, ставится на первую строку после события
ratio, source = pd.Series(np.nan, index=df.index), pd.Series('', index=df.index)
if (DATA / 'splits.csv').exists():
    for s in pd.read_csv(DATA / 'splits.csv', parse_dates=['tradedate']).itertuples():
        near = df.index[(df['SECID'] == s.secid) & (df['TRADEDATE'] - s.tradedate).abs().le(pd.Timedelta(days=7))]
        lj = log_jump[near].abs().dropna()
        if len(lj):                                     # событие ставим на строку с максимальным скачком рядом с датой ISS
            j = lj.idxmax(); ratio[j], source[j] = s.after / s.before, 'ISS'
NICE = np.array(sorted({k * 10.0 ** e for k in (1, 2, 3, 4, 5) for e in range(-6, 6)}))
unexplained = (log_jump.abs() > np.log(3)) & ratio.isna()   # скачок в 3+ раза за день — не рыночное движение
for j in df.index[unexplained]:
    ratio[j] = NICE[np.abs(np.log(NICE) - (-log_jump[j])).argmin()]; source[j] = 'по скачку цены'
f = ratio.fillna(1.0)
factor = f.groupby(df['SECID']).transform(lambda x: x[::-1].cumprod()[::-1]) / f   # произведение коэффициентов событий после строки
for c in ['OPEN', 'HIGH', 'LOW', 'CLOSE']:
    df[f'ADJ_{c}'] = df[c] / factor
df['ADJ_VOLUME'] = df['VOLUME'] * factor
print('сплиты и консолидации:')
print(df.loc[ratio.notna(), ['SECID', 'TRADEDATE']].assign(ratio=ratio, источник=source).to_string(index=False))

# дивиденды: сумма на скорректированную бумагу в день отсечки
df['DIV'] = 0.0
if (DATA / 'dividends.csv').exists():
    dv = pd.read_csv(DATA / 'dividends.csv', parse_dates=['registryclosedate'])
    dv = dv[(dv['currencyid'] == 'RUB') & (dv['value'] > 0) & dv['secid'].isin(set(df['SECID']))]
    n_div = 0
    for d in dv.itertuples():
        ss = np.searchsorted(cal, np.datetime64(d.registryclosedate), 'right')
        k = 1 if cal[min(max(ss - 1, 0), len(cal) - 1)] >= np.datetime64('2023-07-31') else 2
        i = ss - k                                      # первый день, сделки которого рассчитываются после закрытия реестра
        rows = df.index[(df['SECID'] == d.secid) & (df['DAY'] >= i)]
        if 0 <= i < len(cal) and len(rows) and df.at[rows[0], 'DAY'] - i <= 5:
            df.at[rows[0], 'DIV'] += d.value / factor[rows[0]]; n_div += 1
    print(f'дивидендов начислено: {n_div} из {len(dv)} записей ISS')
else:
    print('ВНИМАНИЕ: dividends.csv нет, доходность считается без дивидендов')

g = df.groupby('SECID')
df['RETURN'] = ((df['ADJ_CLOSE'] + df['DIV']) / g['ADJ_CLOSE'].shift(1) - 1).where(gap == 1)   # через пропуск сессии не считаем
assert np.log1p(df['RETURN']).abs().max() < np.log(3), 'после корректировки остались скачки в 3+ раза'
for k in range(5):
    df[f'RET_LAG{k}'] = g['RETURN'].shift(k)
vol_ma = g['ADJ_VOLUME'].transform(lambda x: x.rolling(20, min_periods=5).mean())
df['LOG_VOL_NORM'] = np.log(df['ADJ_VOLUME'].clip(lower=1) / vol_ma.clip(lower=1))
df['FUTURE_RETURN'] = g['RETURN'].shift(-1)                      # NaN, если следующая сессия пропущена
df[['SECID', 'TRADEDATE', 'ADJ_CLOSE', 'ADJ_VOLUME', 'DIV', 'RETURN']].to_csv(DATA / 'prices_adj.csv', index=False)

print(f'доходностей через пропуск сессии: {int((gap > 1).sum())}; '
      f'максимальная |доходность| за день: {df["RETURN"].abs().max():.1%}')
cols = ['SECID', 'TRADEDATE', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME', 'ADJ_CLOSE', 'ADJ_VOLUME', 'DIV', 'RETURN',
        'RET_LAG0', 'RET_LAG1', 'RET_LAG2', 'RET_LAG3', 'RET_LAG4', 'LOG_VOL_NORM', 'FUTURE_RETURN']
df = df.dropna(subset=['FUTURE_RETURN', 'RET_LAG4', 'LOG_VOL_NORM'])
df[cols].reset_index(drop=True).to_csv(DATA / 'targets.csv', index=False)
print(f'{len(df):,} строк, {df.SECID.nunique()} тикеров')

сплиты и консолидации:
SECID  TRADEDATE    ratio источник
 GMKN 2024-04-08 100.0000      ISS
 PLZL 2025-03-27  10.0000      ISS
TRNFP 2024-02-21 100.0000      ISS
 VTBR 2024-07-15   0.0002      ISS
ВНИМАНИЕ: dividends.csv нет, доходность считается без дивидендов


доходностей через пропуск сессии: 8; максимальная |доходность| за день: 25.0%


35,288 строк, 46 тикеров


## 6. Матрица признаков узлов (N × 775)

Узел: пара (тикер, торговый день). Время новостей переводится из UTC в Москву. Новость до закрытия
аукциона 18:50 относится к ближайшей торговой сессии не раньше её даты (новости выходных и праздников
переходят на следующий торговый день), новость после 18:50 — строго к следующей сессии. Эмбеддинг
узла равен взвешенному среднему эмбеддингов привязанных новостей (веса уровней 1,0 / 0,5 / 0,2),
считается через `np.add.at` чанками, чтобы не материализовать матрицу «связи × 768».
Для проверки H2 отдельно сохраняется матрица `node_features_l1.npy`, где эмбеддинг и флаг новостей
считаются только по явным упоминаниям эмитента (уровень L1), без отраслевой и широковещательной привязки.

In [6]:
PRICE_COLS = ['RET_LAG0', 'RET_LAG1', 'RET_LAG2', 'RET_LAG3', 'RET_LAG4', 'LOG_VOL_NORM']
CLOSE_MSK = pd.Timestamp('18:50').time()

links = pd.read_csv(DATA / 'news_linked.csv', usecols=['news_id', 'SECID', 'weight', 'date'], parse_dates=['date'])
targets = pd.read_csv(DATA / 'targets.csv', parse_dates=['TRADEDATE'])
cal = np.sort(pd.read_csv(DATA / 'prices_adj.csv', usecols=['TRADEDATE'], parse_dates=['TRADEDATE'])['TRADEDATE'].unique())
emb = np.load(DATA / 'news_embeddings.npy')
emb_pos = pd.Series(np.arange(len(emb)), index=pd.read_csv(DATA / 'news_embeddings_index.csv')['news_id'].values)

msk = links['date'].dt.tz_convert('Europe/Moscow') if links['date'].dt.tz is not None else links['date'].dt.tz_localize('UTC').dt.tz_convert('Europe/Moscow')
day = msk.dt.normalize().dt.tz_localize(None).values
late = (msk.dt.time >= CLOSE_MSK).to_numpy()
i = np.where(late, np.searchsorted(cal, day, 'right'), np.searchsorted(cal, day, 'left'))   # ближайшая допустимая сессия
links['DATE'] = pd.Series(cal[np.minimum(i, len(cal) - 1)]).where(i < len(cal)).values
links = links.dropna(subset=['DATE']).sort_values(['SECID', 'DATE']).reset_index(drop=True)
print(f'новостей в нерабочие дни, перенесённых на следующую сессию: {int((~late & (i < len(cal)) & (cal[np.minimum(i, len(cal) - 1)] != day)).sum()):,} связей')

base = targets.dropna(subset=PRICE_COLS + ['FUTURE_RETURN']).reset_index(drop=True)


def feature_matrix(lk):
    """Признаки узлов (тикер, день): цены + флаг новостей + взвешенное среднее эмбеддингов связей lk."""
    grp = lk.groupby(['SECID', 'DATE'], sort=False)
    gid = grp.ngroup().to_numpy()
    keys = grp.size().reset_index()[['SECID', 'DATE']].assign(emb_row=np.arange(grp.ngroups))
    w, rows = lk['weight'].to_numpy(np.float32), emb_pos.loc[lk['news_id']].to_numpy()
    sum_e, sum_w = np.zeros((grp.ngroups, emb.shape[1]), np.float32), np.zeros(grp.ngroups, np.float32)
    for s in range(0, len(lk), 100_000):
        sl = slice(s, s + 100_000)
        np.add.at(sum_e, gid[sl], emb[rows[sl]] * w[sl, None]); np.add.at(sum_w, gid[sl], w[sl])
    mean_e = sum_e / np.maximum(sum_w[:, None], 1e-12)
    m = base.merge(keys, left_on=['SECID', 'TRADEDATE'], right_on=['SECID', 'DATE'], how='left')
    has = m['emb_row'].notna().to_numpy()
    X = np.zeros((len(m), len(PRICE_COLS) + 1 + emb.shape[1]), np.float32)
    X[:, :6], X[:, 6] = m[PRICE_COLS].to_numpy(np.float32), has
    X[has, 7:] = mean_e[m.loc[has, 'emb_row'].to_numpy(int)]
    assert np.isfinite(X).all()
    return m, has, X


m, has, X = feature_matrix(links)
np.save(DATA / 'node_features.npy', X)
m[['SECID', 'TRADEDATE', 'FUTURE_RETURN']].assign(has_news=has.astype(np.int8)).to_csv(DATA / 'node_features_index.csv', index=False)
_, has_l1, X_l1 = feature_matrix(links[links['weight'] == 1.0])   # только явные упоминания эмитента (уровень L1)
np.save(DATA / 'node_features_l1.npy', X_l1)
print(X.shape, f'has_news = 1 на {has.mean():.1%} строк; явное упоминание эмитента (L1) есть у {has_l1.mean():.1%} строк')

новостей в нерабочие дни, перенесённых на следующую сессию: 50,257 связей


(35256, 775) has_news = 1 на 99.9% строк; явное упоминание эмитента (L1) есть у 39.6% строк


## 7. Динамический граф эмитентов

Минимальное остовное дерево (Mantegna, 1999) на 60-дневном окне корреляций остатков рыночной
модели: для каждой бумаги внутри окна оценивается регрессия log-доходности на равновзвешенный
рынок дня, `r_i = a_i + b_i m + e_i`, и корреляции считаются по `e_i`. Без этого почти все пары
положительно скоррелированы и дерево вырождается в звезду; простое вычитание среднего по рынку
(без беты) порождало бы ложные отрицательные связи между бумагами с высокой и низкой бетой.
Доходности берутся скорректированные (стадия 5). Расстояние берётся по |ρ|, знак хранится как
атрибут ребра `[sign ρ, |ρ|]`. Граф считается на последний день месяца и действует весь следующий
месяц; дни до первого графа графа не получают. Отдельно строится статический секторный граф для абляции.

In [7]:
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree

WINDOW, MIN_OBS = 60, 40
prices = pd.read_csv(DATA / 'prices_adj.csv', parse_dates=['TRADEDATE'])
nfi = pd.read_csv(DATA / 'node_features_index.csv', parse_dates=['TRADEDATE'])
tickers = pd.read_csv(DATA / 'tickers.csv')
nodes = (tickers[tickers['SECID'].isin(nfi['SECID'].unique())][['SECID', 'SHORTNAME', 'SECTOR']]
         .drop_duplicates('SECID').sort_values('SECID').reset_index(drop=True))
nodes.insert(0, 'node_id', nodes.index)
nid = dict(zip(nodes['SECID'], nodes['node_id']))

ret = np.log1p(prices.pivot(index='TRADEDATE', columns='SECID', values='RETURN').sort_index().reindex(columns=nodes['SECID']))
month_ends = pd.Series(ret.index, index=ret.index).groupby(ret.index.to_period('M')).max()


def residuals(w):
    """Остатки регрессии каждой бумаги на равновзвешенный рынок внутри окна."""
    m, out = w.mean(axis=1), w.copy()
    for c in w.columns:
        ok = (w[c].notna() & m.notna()).values
        b, a = np.polyfit(m.values[ok], w[c].values[ok], 1)
        out[c] = w[c] - a - b * m
    return out


def mst_edges(window):
    valid = window.columns[window.notna().sum() >= MIN_OBS]
    if len(valid) < 2:
        return [], 0
    rho = np.nan_to_num(residuals(window[valid]).corr().values); np.fill_diagonal(rho, 1.0)
    dist = np.sqrt(np.clip(2 * (1 - np.abs(rho)), 0, None))
    mst = minimum_spanning_tree(csr_matrix(np.triu(dist + 1e-9, k=1))).tocoo()
    edges = []
    for u, v in zip(mst.row, mst.col):
        a, b = sorted((nid[valid[u]], nid[valid[v]]))
        r = float(rho[u, v]); edges.append((a, b, int(np.sign(r)) if r != 0 else 1, abs(r)))
    return edges, len(valid)


graphs, meta = {}, []
for me in month_ends:
    p = ret.index.get_loc(me)
    edges, n_active = mst_edges(ret.iloc[max(0, p - WINDOW + 1):p + 1])
    if not edges:
        continue
    bucket = (me.to_period('M') + 1).strftime('%Y-%m')          # действует на следующий месяц
    graphs[bucket] = edges
    meta.append(dict(month_bucket=bucket, window_end=me.date(), n_active_nodes=n_active, n_edges=len(edges),
                     n_pos=sum(e[2] > 0 for e in edges), n_neg=sum(e[2] < 0 for e in edges),
                     mean_abs_corr=np.mean([e[3] for e in edges]), median_abs_corr=np.median([e[3] for e in edges])))
meta = pd.DataFrame(meta)
assert (meta['n_edges'] == meta['n_active_nodes'] - 1).all()      # дерево: рёбер = узлов − 1

nodes.to_csv(DATA / 'graph_nodes.csv', index=False)
pd.DataFrame([(mb, *e) for mb, es in graphs.items() for e in es],
             columns=['month_bucket', 'src', 'dst', 'sign', 'abs_corr']).sort_values(['month_bucket', 'src', 'dst']).to_csv(DATA / 'graph_edges_corr.csv', index=False)
ei, ea = {}, {}
for mb, es in graphs.items():                                     # PyG-формат: обе ориентации, [прямые; обратные]
    src, dst, sg, ac = map(np.array, zip(*es))
    ei[mb] = np.stack([np.r_[src, dst], np.r_[dst, src]]).astype(np.int64)
    ea[mb] = np.stack([np.r_[sg, sg], np.r_[ac, ac]], 1).astype(np.float32)
np.savez(DATA / 'edge_index_corr.npz', **ei); np.savez(DATA / 'edge_attr_corr.npz', **ea)
meta.to_csv(DATA / 'graph_meta.csv', index=False)

sec = pd.DataFrame([dict(src=a, dst=b, sector=s) for s, g in nodes[nodes['SECTOR'] != 'Other'].groupby('SECTOR')
                    for a, b in itertools.combinations(sorted(g['node_id']), 2)])
sec.to_csv(DATA / 'graph_edges_sector.csv', index=False)
np.save(DATA / 'edge_index_sector.npy', np.stack([np.r_[sec.src, sec.dst], np.r_[sec.dst, sec.src]]).astype(np.int64))

dates = pd.DatetimeIndex(sorted(nfi['TRADEDATE'].unique()))
d2m = pd.DataFrame({'TRADEDATE': dates.strftime('%Y-%m-%d'), 'month_bucket': dates.strftime('%Y-%m')})
d2m[d2m['month_bucket'].isin(graphs)].to_csv(DATA / 'date_to_month.csv', index=False)   # дни до первого графа без графа
print(f'{len(graphs)} месячных графов, доля отрицательных рёбер {meta.n_neg.sum() / meta.n_edges.sum():.3f}, '
      f'секторных рёбер {len(sec)}')

40 месячных графов, доля отрицательных рёбер 0.520, секторных рёбер 138
